<a href="https://colab.research.google.com/github/GiovanniPasq/agentic-rag-for-dummies/blob/main/notebooks/observability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Observability for Agentic RAG Systems

This guide is a conceptual appendix for the project. It explains **why observability matters** for agentic RAG, how this repo wires optional **Langfuse** tracing into LangGraph, and when you might reach for a dedicated observability platform.

For quantitative RAG quality scoring, use the companion [evaluation notebook](evaluation.ipynb). That notebook owns the QA dataset loading and RAGAS metric execution flow.

## 1. Why Observability Matters

Traditional applications fail loudly: an HTTP 500, a stack trace, a timeout. LLM-based systems fail **silently**. An agent can select the wrong tool, retrieve stale documents, hallucinate a correlation, and present the result confidently - all without raising a single exception.

Observability for agentic systems answers questions that standard APM tools cannot:

- **Did the retriever surface the right chunks?** (Context Precision / Recall)
- **Is the generated answer grounded in the retrieved data?** (Faithfulness)
- **Which tool did the agent call, and was it the right choice?** (Tool Call tracing)
- **Where is latency accumulating across the graph?** (Span-level timing)
- **How much does each query cost?** (Token-level cost tracking)

## 2. Observability vs Evaluation

Observability and evaluation answer related but different questions:

| Question | Best tool |
|---|---|
| What happened during this user request? | Observability / tracing |
| Which graph node or tool call caused latency or failure? | Observability / tracing |
| Did the retriever return relevant chunks? | Evaluation metrics |
| Was the final response accurate and grounded? | Evaluation metrics |

The evaluation notebook uses exactly five RAGAS metrics:

| Dimension | Metric | What it measures |
|---|---|---|
| **Generation** | [Answer Accuracy](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/nvidia_metrics/#answer-accuracy) | Agreement between the generated answer and the reference answer |
| **Retrieval** | [Context Relevance](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/nvidia_metrics/#context-relevance) | Whether retrieved chunks are pertinent to the query |
| **Generation** | [Response Groundedness](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/nvidia_metrics/#response-groundedness) | Whether response claims are supported by retrieved contexts |
| **Retrieval** | [Context Precision](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/context_precision/) | Whether relevant chunks are ranked ahead of irrelevant chunks |
| **Retrieval** | [Context Recall](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/context_recall/) | Whether retrieved contexts cover the reference answer |

Use traces to debug individual runs; use evaluation datasets to compare retrieval thresholds, embedding models, prompts, and model choices over many runs.

## 3. Telemetry Anatomy: Sessions, Traces, and Spans

Agentic systems produce **hierarchical, non-linear execution trees**. The standard telemetry model breaks down into three levels:

- **Session** - A full multi-turn conversation. Tracks memory coherence and behavioral drift over time.
- **Trace** - One complete request-response cycle within a session. Contains timing and cost data.
- **Span** - An atomic unit of work inside a trace: an LLM call, a tool invocation, a retrieval step.

In an agentic graph like this project's, a single user question can produce dozens of spans: query rewriting, parallel sub-agent invocations, multiple search/retrieve cycles, context compression, and final aggregation.

## 4. Platform Comparison

For this educational repo, Langfuse is enough: it is open-source, works well with LangGraph callbacks, and can be self-hosted. Other platforms are worth knowing about, but pricing, licenses, and free tiers change often, so check each project directly before making a production choice.

| Platform | Primary strength | Typical fit |
|---|---|---|
| [Langfuse](https://langfuse.com) | Open-source tracing and prompt management | Self-hosted or cloud tracing for LLM apps |
| [LangSmith](https://smith.langchain.com) | LangChain/LangGraph-native debugging | Deep graph debugging in the LangChain ecosystem |
| [Arize Phoenix](https://phoenix.arize.com) | ML/LLM observability and embedding analysis | Retrieval quality, drift, and dataset analysis |
| [AgentOps](https://agentops.ai) | Agent run monitoring | Tool-heavy and multi-agent workflows |
| [Braintrust](https://braintrust.dev) | Evaluation and CI workflows | Regression testing and production eval loops |
| [Helicone](https://helicone.ai) | AI gateway/proxy observability | Provider-level logging, caching, and cost controls |

### Integration approaches

These tools generally use one of three strategies:

1. **SDK-based instrumentation**: decorators or wrappers in your code.
2. **OpenTelemetry/callback instrumentation**: spans are emitted from framework hooks.
3. **Proxy/gateway instrumentation**: requests are captured at the network/provider boundary.

This project uses the callback approach: `RAGSystem.get_config()` injects a Langfuse handler into the LangGraph config, and LangGraph propagates it through nodes, subgraphs, tool calls, and LLM calls.

## 5. Practical Integration: Langfuse with this Project

This project uses Langfuse because it's open-source, framework-agnostic, and trivial to integrate with LangGraph via the callback system.

### How it works

LangGraph propagates `callbacks` from the top-level `graph.stream()` config down to every node, subgraph, and LLM call automatically. This means we only need to inject the Langfuse callback handler in **one place** — the `get_config()` method — and every operation gets traced.

```
graph.stream(input, config={"callbacks": [langfuse_handler]})
    └─ summarize_history   → llm.invoke() ← traced
    └─ rewrite_query       → llm.invoke() ← traced
    └─ agent (subgraph, via Send())
    │   └─ orchestrator    → llm.invoke() ← traced
    │   └─ tools           → search/retrieve ← traced
    │   └─ compress_context → llm.invoke() ← traced
    └─ aggregate_answers   → llm.invoke() ← traced
```


### Setup

**Step 1** — Install dependencies:

```bash
pip install -r requirements.txt
```

`langfuse` is already included in the project requirements.

**Step 2** — Configure environment variables.

For the modular app, copy the template and edit it:

```bash
cp project/.env.example project/.env
```

Then set:

```bash
LANGFUSE_ENABLED=true
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_BASE_URL=https://cloud.langfuse.com  # or your self-hosted URL
```

**Step 3** — Run the app normally:

```bash
python project/app.py
```

To disable tracing, set `LANGFUSE_ENABLED=false` or leave it unset. The app runs the same way without tracing.

### The integration code

The full implementation lives in three files. Here's what each one does:

In [ ]:
# project/core/observability.py — simplified view

import config

class Observability:
    def __init__(self):
        self._enabled = config.LANGFUSE_ENABLED
        self._handler = None
        self._client = None

        if not self._enabled:
            return

        from langfuse import get_client
        from langfuse.langchain import CallbackHandler

        self._client = get_client()
        if self._client.auth_check():
            self._handler = CallbackHandler()
        else:
            self._enabled = False

    def get_handler(self):
        return self._handler

    def flush(self):
        if self._client is not None:
            self._client.flush()

# The real implementation also catches configuration/import errors and falls
# back gracefully when Langfuse is disabled or credentials are missing.

In [ ]:
# core/rag_system.py

from core.observability import Observability

class RAGSystem:
    def __init__(self):
        self.vector_db = VectorDbManager()
        self.parent_store = ParentStoreManager()
        self.chunker = DocumentChunker()
        self.observability = Observability()  
        # ...

    def get_config(self):
        cfg = {
            "configurable": {"thread_id": self.thread_id},
            "recursion_limit": self.recursion_limit,
        }
        handler = self.observability.get_handler()
        if handler:
            cfg["callbacks"] = [handler]
        return cfg

# LangGraph propagates callbacks to ALL nodes and subgraphs automatically.

In [ ]:
# project/core/chat_interface.py — flush pending traces when the session is cleared

def clear_session(self):
    self.rag_system.reset_thread()
    self.rag_system.observability.flush()

### What gets traced

With this setup, Langfuse captures:

- Every LLM call across all graph nodes (orchestrator, query rewriting, summarization, compression, aggregation)
- Structured output parsing (e.g. `QueryAnalysis` in the rewrite step)
- Tool invocations (`search_child_chunks`, `retrieve_parent_chunks`) with arguments and results
- The full graph execution flow including parallel sub-agent fan-out
- Token counts and latency per span

## 6. Langfuse Hosting Options

### Option A: Langfuse Cloud

Create a project at [cloud.langfuse.com](https://cloud.langfuse.com), copy the API keys, and set them in `project/.env`.

### Option B: Self-hosted

For data-sovereign deployments, run the official Langfuse stack and point `LANGFUSE_BASE_URL` to your instance:

```bash
git clone https://github.com/langfuse/langfuse.git
cd langfuse
docker compose up
```

Then set:

```bash
LANGFUSE_ENABLED=true
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_BASE_URL=http://localhost:3000
```

See the [official self-hosting documentation](https://langfuse.com/self-hosting) for production configuration.

## 7. Further Reading

- [Langfuse documentation](https://langfuse.com/docs)
- [Langfuse LangChain/LangGraph integration](https://docs.langchain.com/oss/python/integrations/providers/langfuse)
- [LangGraph streaming](https://langchain-ai.github.io/langgraph/how-tos/streaming/)
- [RAGAS evaluation framework](https://docs.ragas.io/)
- [OpenTelemetry semantic conventions for GenAI](https://opentelemetry.io/docs/specs/semconv/gen-ai/)